# Day 53: Simulate an Autoscaling Policy

**Layer:** Production


## Concept Overview

An autoscaling policy maps observed metrics (concurrency, queue depth, GPU util) to replica count decisions. This simulation implements a reactive policy with hysteresis to prevent thrashing.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Reactive Autoscaling with Hysteresis

An autoscaling policy maps observed metrics (concurrency, queue depth, GPU util) to replica count de...


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class HysteresisAutoscaler:
    def __init__(self, min_r=1, max_r=16, target_conc=8, scale_up_threshold=0.9,
                 scale_down_threshold=0.5, cooldown_steps=20):
        self.replicas = min_r
        self.min_r = min_r; self.max_r = max_r
        self.target = target_conc
        self.up_thresh = scale_up_threshold
        self.dn_thresh = scale_down_threshold
        self.cooldown = cooldown_steps
        self.last_scale = -cooldown_steps

    def step(self, t, concurrency):
        util = concurrency / (self.replicas * self.target)
        if util > self.up_thresh and t - self.last_scale > self.cooldown:
            self.replicas = min(self.max_r, self.replicas + 1)
            self.last_scale = t
        elif util < self.dn_thresh and t - self.last_scale > self.cooldown * 2:
            self.replicas = max(self.min_r, self.replicas - 1)
            self.last_scale = t
        return self.replicas

np.random.seed(42)
T = 200
conc = np.concatenate([np.random.poisson(5, 50), np.random.poisson(20, 50),
                        np.random.poisson(8, 50), np.random.poisson(30, 50)])
scaler = HysteresisAutoscaler()
replicas = [scaler.step(t, conc[t]) for t in range(T)]

fig, (ax1,ax2) = plt.subplots(2,1,figsize=(12,6),sharex=True)
ax1.plot(conc, label="Concurrency"); ax1.set_ylabel("Active Requests"); ax1.legend()
ax2.step(range(T), replicas, where="post", color="green", label="Replicas")
ax2.set_ylabel("Replica Count"); ax2.set_xlabel("Time"); ax2.legend()
plt.tight_layout(); plt.savefig("autoscaler.png",dpi=100); plt.show()
print(f"Scale events: {sum(1 for i in range(1,T) if replicas[i]!=replicas[i-1])}")

## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- An autoscaling policy maps observed metrics (concurrency, queue depth, GPU util) to replica count decisions.
- Scale-up threshold (90%) must be lower than 100% to leave headroom — if you wait until saturation to scale, queues have already built up. Scale-down needs 2x longer cooldown to prevent thrashing..
- Day 53 implementation complete.
